# Python, Семинар 6.

**Важно:** автор (Sergey Korpachev) данного семинара не является специалистом в ROS 2 и робототехнике. Поэтому если у вас есть какие-либо комментарии / замечения / предложения по данному файлу (коду, логике и прочему), то можете смело их написать на почту (sskorpachev@gmail.com) или в Tg (кто знает).

Спасибо :)

**Внимание:** коды ниже являются только демо-версией реального ROS 2. Это было сделано с той целью, чтобы хоть как-то адекватно запустить ROS 2 и его инфраструктура в Google Colab.

Сначала загрузим необходимые библиотеки.

In [ ]:
import asyncio
import random
from collections import defaultdict
from typing import Callable, Any

## Задача 1. Робот-пылесос и стена.

### Ситуация.

Вы разрабатываете прошивку для робота-пылесоса. У робота есть **ультразвуковой датчик расстояния**, который измеряет, сколько метров осталось до ближайшего препятствия (стены или мебели). Датчик "смотрит" вперёд и обновляет измерение несколько раз в секунду.

Отдельно от датчика работает **модуль безопасности**. Его задача - следить за расстоянием и подавать сигнал тревоги, если робот подобрался к препятствию слишком близко (меньше **1 метра**). Если расстояние больше - модуль просто подтверждает, что всё в порядке.

**Проблема:** датчик и модуль безопасности - это две разные программы (в реальном роботе - два разных процесса). Они не должны знать друг о друге напрямую: датчик не вызывает функции модуля безопасности, а модуль не обращается к "внутренностям" датчика. Иначе при замене датчика придётся переписывать модуль безопасности, и наоборот.

**Решение:** они общаются через **топик** - именованный канал, в который датчик "сообщает" свои измерения, а модуль безопасности "слушает". Ровно так это устроено в ROS 2.

### Что нужно сделать?

Вам дана готовая инфраструктура "мини-ROS" (классы `MiniROS`, `Publisher`, `Subscription`, `Node`) - её писать не нужно, только прочитать и понять, как она работает.

Ваша задача - заполнить два класса-заготовки:

#### 1. `SensorNode` - нода-датчик.

- В конструкторе создайте паблишер (публикатор) на топик `"distance"`.
- В методе `run()` в бесконечном цикле:
  - сгенерируйте случайное расстояние от **0.1 до 10.0 метров** (округлите до 2 знаков);
  - опубликуйте его в топик в виде словаря `{"distance_m": <значение>}`;
  - подождите `self.frequency` секунд перед следующим измерением.

#### 2. `LoggerNode` - нода-модуль безопасности.

- В конструкторе создайте подписку на топик `"distance"` с обработчиком `self.callback`.
- В методе `callback(msg)`:
  - достаньте расстояние из сообщения (`msg["distance_m"]`);
  - если расстояние **меньше 1.0 м** - напечатайте предупреждение об опасном сближении;
  - иначе - напечатайте, что расстояние в норме.


### Как проверить, что получилось?

Запустите последнюю ячейку с `await main()`. Если всё сделано правильно, вы увидите в логах примерно такую картину:

```text
[Node Init] Создана нода: sensor_node
[Node Init] Создана нода: logger_node
[Publisher -> distance] Отправка: {'distance_m': 7.4}
[logger_node <- distance] Получено: {'distance_m': 7.4}
[OK] Дистанция в норме: 7.4 м
[Publisher -> distance] Отправка: {'distance_m': 9.72}
[logger_node <- distance] Получено: {'distance_m': 9.72}
[OK] Дистанция в норме: 9.72 м
[Publisher -> distance] Отправка: {'distance_m': 7.06}
[logger_node <- distance] Получено: {'distance_m': 7.06}
[OK] Дистанция в норме: 7.06 м
[Publisher -> distance] Отправка: {'distance_m': 8.3}
[logger_node <- distance] Получено: {'distance_m': 8.3}
[OK] Дистанция в норме: 8.3 м

[Main] Завершение работы (shutdown).
```

Обратите внимание: `SensorNode` нигде не упоминает `LoggerNode`, а `LoggerNode` не упоминает `SensorNode`. Их связывает только имя топика `"distance"` и формат сообщения.

Готовый код:

In [ ]:
class MiniROS:
    """Имитация ROS 2. Управляет топиками и подписками.
    Это замена DDS-слоя (DDS - Data Distribution Service)."""

    def __init__(self):
        self._topics = defaultdict(list)

    def create_publisher(self, topic_name: str):
        return Publisher(self, topic_name)

    def create_subscription(self, topic_name: str, callback: Callable):
        self._topics[topic_name].append(callback)
        return Subscription(self, topic_name, callback)

    async def _publish(self, topic_name: str, msg: Any):
        await asyncio.sleep(0.05)  # имитация задержки сети
        for callback in self._topics[topic_name]:
            callback(msg)


class Publisher:
    """Обёртка над топиком для отправки сообщений."""

    def __init__(self, ros: MiniROS, topic: str):
        self.ros = ros
        self.topic = topic

    async def publish(self, msg: Any):
        print(f"[Publisher -> {self.topic}] Отправка: {msg}")
        await self.ros._publish(self.topic, msg)


class Subscription:
    """Хэндл подписки."""

    def __init__(self, ros: MiniROS, topic: str, callback: Callable):
        self.ros = ros
        self.topic = topic
        self.callback = callback


class Node:
    """Базовый класс ноды. Аналог rclpy.node.Node."""

    def __init__(self, name: str, ros: MiniROS):
        self.name = name
        self.ros = ros
        print(f"[Node Init] Создана нода: {self.name}")

    def create_publisher(self, topic_name: str):
        return self.ros.create_publisher(topic_name)

    def create_subscription(self, topic_name: str, callback: Callable):
        def wrapped_callback(msg):
            print(f"[{self.name} <- {topic_name}] Получено: {msg}")
            callback(msg)
        return self.ros.create_subscription(topic_name, wrapped_callback)

Нужно написать:

In [ ]:
# Подсказки:
#   - паблишер создаётся через self.create_publisher("имя_топика");
#   - подписка создаётся через self.create_subscription("имя_топика", self.callback);
#   - публикация: await self.pub.publish({...});
#   - не забывайте await asyncio.sleep(...) в цикле.


class SensorNode(Node):
    """Нода-датчик: генерирует случайное расстояние и публикует его."""
    def __init__(self, ros: MiniROS, name: str = "sensor_node", frequency: float = 1.0):
        super().__init__(name, ros)
        # TODO 1: создайте паблишер на топик "distance"
        # self.pub = ...
        self.pub = None  # <-- замените
        self.frequency = frequency

    async def run(self):
        """Бесконечный цикл публикации."""
        while True:
            # TODO 2: сгенерируйте случайное расстояние от 0.1 до 10.0
            #         (округлите до 2 знаков)
            # distance = ...
            distance = None  # <-- замените

            # TODO 3: опубликуйте словарь {"distance_m": distance}
            # await self.pub.publish(...)
            pass  # <-- замените

            # TODO 4: подождите self.frequency секунд
            # await asyncio.sleep(...)
            pass  # <-- замените


class LoggerNode(Node):
    """Нода-логгер: подписывается на "distance" и предупреждает об опасности."""

    def __init__(self, ros: MiniROS, name: str = "logger_node"):
        super().__init__(name, ros)
        # TODO 5: создайте подписку на "distance" с обработчиком self.callback
        # self.sub = ...
        self.sub = None  # <-- замените

    def callback(self, msg):
        """Вызывается при получении сообщения из топика."""
        # TODO 6: достаньте расстояние из словаря
        # dist = ...
        dist = None  # <-- замените

        # TODO 7: если dist < 1.0 - печатайте предупреждение,
        #         иначе - сообщение "OK"
        # if dist < 1.0:
        #     print(...)
        # else:
        #     print(...)
        pass  # <-- замените

Решение:

In [ ]:
# Подсказки:
#   - паблишер создаётся через self.create_publisher("имя_топика");
#   - подписка создаётся через self.create_subscription("имя_топика", self.callback);
#   - публикация: await self.pub.publish({...});
#   - не забывайте await asyncio.sleep(...) в цикле.


### ╰( ͡° ͜ʖ ͡° )つ──☆*:・ﾟ


Запуск:

In [ ]:
# Это аналог rclpy.spin(): запускает ноды и даёт им поработать.

async def main():
    ros = MiniROS()
    sensor = SensorNode(ros, frequency=1.5)
    logger = LoggerNode(ros)

    task = asyncio.create_task(sensor.run())
    await asyncio.sleep(6)
    task.cancel()
    print("\n[Main] Завершение работы (shutdown).")


await main()

## Асинхронная работа в Python.

Это тема для отдельных лекции и семинара, но давайте чуть-чуть поговорим про то, как можно организовать асинхронную работу в Python. Про корутины и подобное. Так что рекомендуем отдельно изучить потоки / процессы / параллелизм.

*Данные примеры писались и запускались в Google Colab, хотя это не самая благодарная работа. У Colab есть свой Event Loop, с которым могут быть сложности (например, при создании Tg-ботов).*

**Что такое корутина?**

**Корутина (coroutine)** - сокращение от cooperative routine ("кооперативная подпрограмма").

**Routine** - "подпрограмма", то есть функция. В обычном Python любая функция - это routine: вызвал -> выполнилась от начала до конца -> вернула результат.

**Cooperative** - "кооперативная", то есть сама уступает управление. Это ключевое слово. В отличие от потоков (threads), где операционная система принудительно переключает задачи (preemptive multitasking), корутина сама решает, когда отдать управление - через `await`. Она "кооперируется" с event loop'ом: "я подожду, а ты пока займись другими".

### Примеры:

#### Пример 1. Синхронный код (без async).

In [ ]:
import time

def make_coffee():
    print("Начинаю варить кофе...")
    time.sleep(2)  # блокирует всё
    print("Кофе готов")

def make_toast():
    print("Начинаю делать тост...")
    time.sleep(2)  # блокирует всё
    print("Тост готов")

# Последовательно: 2 + 2 = 4 секунды
start = time.time()
make_coffee()
make_toast()
print(f"Итого: {time.time() - start:.1f} с")

**Вывод:** пока варится кофе, тост не начинается. Всё блокируется.

#### Пример 2. Асинхронный код (с async / await).

In [ ]:
import asyncio

async def make_coffee_async():
    print("Начинаю варить кофе...")
    await asyncio.sleep(2)  # не блокирует
    print("Кофе готов")

async def make_toast_async():
    print("Начинаю делать тост...")
    await asyncio.sleep(2)  # не блокирует
    print("Тост готов")

async def main():
    # Запускаем обе задачи параллельно
    await asyncio.gather(
        make_coffee_async(),
        make_toast_async()
    )

# Параллельно: max(2, 2) = 2 секунды
start = time.time()
await main()
print(f"Итого: {time.time() - start:.1f} с")

**Вывод:** кофе и тост готовятся одновременно. Всё занимает 2 секунды, а не 4.

#### Пример 3. `async def` без `await` - валидно, но не асинхронно.

In [ ]:
async def quick_task():
    print("Я корутина, но не жду")
    return 42

async def main():
    result = await quick_task()  # работает, но без паузы
    print(f"Результат: {result}")

await main()

**Вывод:** функция асинхронная по объявлению, но выполняется мгновенно, как обычная.

**Зачем так делать:** иногда нужна единообразная сигнатура (например, все callback'и в интерфейсе - async), даже если конкретная реализация не ждёт. Или подобное.

#### Пример 4. `await` без `async def` - ошибка :(

Следующая ячейка намеренно содержит ошибку. Запустите её - и увидите SyntaxError. Это нормально.

In [ ]:
# Этот код не запустится - SyntaxError :(

def broken():
    await asyncio.sleep(1)  # SyntaxError: 'await' outside async function

**Вывод:** `await` можно писать только внутри `async def`.

#### Пример 5. `asyncio.create_task` - запуск в фоне.

In [ ]:
async def background_worker():
    for i in range(3):
        print(f"[фон] шаг {i}")
        await asyncio.sleep(1)

async def main():
    task = asyncio.create_task(background_worker())
    print("[main] запустил фоновую задачу")
    await asyncio.sleep(4)  # main работает, фон - параллельно
    task.cancel()
    print("[main] остановил фоновую задачу")

await main()

**Вывод:** `create_task` позволяет `main` продолжать работу, пока фоновая задача крутится.

#### Пример 6. Комментарий к задаче 1: сенсор и логгер.

In [ ]:
async def sensor():
    for i in range(3):
        distance = round(random.uniform(0.1, 10.0), 2)
        print(f"[sensor] измерение {i}: {distance} м")
        await asyncio.sleep(1.5)

async def logger():
    for i in range(3):
        await asyncio.sleep(1.5)
        print(f"[logger] получил измерение {i}")

async def main():
    await asyncio.gather(sensor(), logger())

await main()

**Важно:**

Порядок строк зависит от того, кто проснулся первым. В реальной задаче (Задача 1) logger.callback вызывается из _publish, поэтому вывод строго парный: сначала "Отправка", потом "Получено".

**Вывод:** сенсор и логгер работают параллельно, каждый в своём ритме.

### Ключевые моменты:

1. `async def` объявляет корутину - функцию, которую можно приостановить.
2. `await` ставится **только внутри** `async def`.
3. `await` говорит event loop'у: "я жду, займись другими задачами".
4. `async def` без `await` - валидно, но не даёт параллельности.
5. `asyncio.create_task` запускает корутину в фоне.
6. В Colab `await main()` работает на верхнем уровне - ячейка неявно оборачивается в `async def`.

### Дополнительный пример: генератор vs корутина - в чём разница?

In [ ]:
# Генератор: "дай значение - я подожду".

def counter():
    i = 0
    while i < 3:
        yield i  # отдаём значение и замираем
        i += 1

gen = counter()
print(next(gen))
print(next(gen))
print(next(gen))

**Кто управляет:** вы, вызывая `next()`.

**Зачем замирает:** чтобы отдать очередное значение.

**Что делает после пробуждения:** продолжает с того же `yield`.

In [ ]:
# Корутина: "я жду - ты работай".

import asyncio

async def sensor():
    for i in range(3):
        print(f"[sensor] измерение {i}")
        await asyncio.sleep(1)  # отдаём управление и замираем
        print(f"[sensor] измерение {i} отправлено")

async def main():
    task = asyncio.create_task(sensor())
    print("[main] запустил сенсор, занимаюсь своими делами")
    await asyncio.sleep(2.5)
    task.cancel()

await main()

**Кто управляет:** event loop.

**Зачем замирает:** чтобы подождать I/O или таймер.

**Что делает после пробуждения:** продолжает с того же `await`.

#### Сравнение генератора и корутины.

| | Генератор | Корутина |
|---|---|---|
| Синтаксис | `def` + `yield` | `async def` + `await` |
| Кто управляет | Вызывающий (`next`) | Event loop |
| Зачем замирает | Отдать значение | Подождать I/O / таймер |
| Можно в цикле | Да | Да |
| Асинхронность | Нет | Да |
| Аналогия | Конвейер, выдающий детали | Официант, ждущий заказ |

**Важно:**

*"Генератор - это родственник корутины, у них общий механизм приостановки, но в современном Python это разные инструменты."*

За деталями (почему так или не так) изучайте темы в Computer Science или где-то ещё.

## Задача 2. Аварийный стоп.

### Ситуация.

Вы добавили в робота-пылесоса **модуль аварийного стопа**. Он тоже слушает топик `"distance"`, но реагирует **жёстче**: если расстояние меньше **0.5 метра** - публикует в новый топик `"emergency_stop"` сообщение `{"stop": True}`. Если расстояние больше - публикует `{"stop": False}`.

Модуль безопасности (`LoggerNode`) при этом **не меняется** - он продолжает работать как раньше.

### Что нужно сделать?

1. Напишите класс `EmergencyStopNode(Node)`:
   - в конструкторе создайте **подписку** на `"distance"` и **паблишер** на `"emergency_stop"`;
   - в `callback(msg)` достаньте расстояние; если `< 0.5` - опубликуйте `{"stop": True}` и напечатайте `"[EMERGENCY] СТОП!"`; иначе - опубликуйте `{"stop": False}` и напечатайте `"[EMERGENCY] Едем дальше"`.
2. В `main()` создайте `EmergencyStopNode` рядом с существующими нодами.
3. Запустите и убедитесь, что **обе** ноды (`LoggerNode` и `EmergencyStopNode`) получают сообщения из одного топика.


### Ожидаемый результат.

В логах видно, что на каждое сообщение из `"distance"` реагируют **две** ноды:
```
[Node Init] Создана нода: sensor_node
[Node Init] Создана нода: logger_node
[Node Init] Создана нода: emergency_node
[Publisher -> distance] Отправка: {'distance_m': 5.96}
[logger_node <- distance] Получено: {'distance_m': 5.96}
[OK] Дистанция в норме: 5.96 м
[emergency_node <- distance] Получено: {'distance_m': 5.96}
[EMERGENCY] Едем дальше
[Publisher -> emergency_stop] Отправка: {'stop': False}
[Publisher -> distance] Отправка: {'distance_m': 9.87}
[logger_node <- distance] Получено: {'distance_m': 9.87}
[OK] Дистанция в норме: 9.87 м
[emergency_node <- distance] Получено: {'distance_m': 9.87}
[EMERGENCY] Едем дальше
[Publisher -> emergency_stop] Отправка: {'stop': False}
[Publisher -> distance] Отправка: {'distance_m': 7.23}
[logger_node <- distance] Получено: {'distance_m': 7.23}
[OK] Дистанция в норме: 7.23 м
[emergency_node <- distance] Получено: {'distance_m': 7.23}
[EMERGENCY] Едем дальше
[Publisher -> emergency_stop] Отправка: {'stop': False}
[Publisher -> distance] Отправка: {'distance_m': 8.7}
[logger_node <- distance] Получено: {'distance_m': 8.7}
[OK] Дистанция в норме: 8.7 м
[emergency_node <- distance] Получено: {'distance_m': 8.7}
[EMERGENCY] Едем дальше
[Publisher -> emergency_stop] Отправка: {'stop': False}

[Main] Завершение работы (shutdown).
```

**Внимание (может быть актуально):**

Обратите внимание: у LoggerNode порог 1.0 м (предупреждение), а у EmergencyStopNode - 0.5 м (аварийный стоп). Это два разных модуля с разной чувствительностью: логгер предупреждает заранее, а стоп срабатывает только в критической ситуации.

Нужно написать:

In [ ]:
# Добавьте третью ноду EmergencyStopNode, которая:
#   - подписывается на топик "distance";
#   - если расстояние < 0.5 м - публикует в топик "emergency_stop"
#     сообщение {"stop": True} и печатает "[EMERGENCY] СТОП!";
#   - иначе - публикует {"stop": False} и печатает "[EMERGENCY] Едем дальше".
#
# ВАЖНО: callback синхронный, а publish - асинхронный.
# Поэтому внутри callback нельзя писать await self.pub.publish(...).
# Используйте asyncio.create_task(self.pub.publish(...)).
#
# Подсказки:
#   - паблишер создаётся через self.create_publisher("имя_топика");
#   - подписка создаётся через self.create_subscription("имя_топика", self.callback);
#   - публикация в фоне: asyncio.create_task(self.pub.publish({...}));
#   - сохраняйте ссылку на задачу (например, self._tasks.append(task)),
#     чтобы сборщик мусора не убил её до завершения.


class EmergencyStopNode(Node):
    """Нода аварийного стопа: публикует команду остановки при опасном сближении."""

    def __init__(self, ros: MiniROS, name: str = "emergency_node"):
        super().__init__(name, ros)
        # TODO 1: создайте подписку на "distance" с обработчиком self.callback
        # self.sub = ...
        self.sub = None  # <-- замените

        # TODO 2: создайте паблишер на "emergency_stop"
        # self.pub = ...
        self.pub = None  # <-- замените
        self._tasks = []  # храним ссылки на задачи (защита от GC)

    def callback(self, msg):
        """Вызывается при получении сообщения из топика "distance"."""
        # TODO 3: достаньте расстояние из словаря
        # dist = ...
        dist = None  # <-- замените

        # TODO 4: определите, нужен ли стоп, и опубликуйте решение в фоне.
        #   - если dist < 0.5: напечатайте "[EMERGENCY] СТОП!" и задайте stop = True
        #   - иначе: напечатайте "[EMERGENCY] Едем дальше" и задайте stop = False
        #   - затем ОДИН раз опубликуйте {"stop": stop} через create_task
        #     и сохраните задачу в self._tasks (см. подсказки выше)
        # if dist < 0.5:
        #     print(...)
        #     stop = True
        # else:
        #     print(...)
        #     stop = False
        # task = asyncio.create_task(self.pub.publish({"stop": stop}))
        # self._tasks.append(task)
        pass  # <-- замените

Решение:

In [ ]:
# Добавьте третью ноду EmergencyStopNode, которая:
#   - подписывается на топик "distance";
#   - если расстояние < 0.5 м - публикует в топик "emergency_stop"
#     сообщение {"stop": True} и печатает "[EMERGENCY] СТОП!";
#   - иначе - публикует {"stop": False} и печатает "[EMERGENCY] Едем дальше".
#
# ВАЖНО: callback синхронный, а publish - асинхронный.
# Поэтому внутри callback нельзя писать await self.pub.publish(...).
# Используйте asyncio.create_task(self.pub.publish(...)).
#
# Подсказки:
#   - паблишер создаётся через self.create_publisher("имя_топика");
#   - подписка создаётся через self.create_subscription("имя_топика", self.callback);
#   - публикация в фоне: asyncio.create_task(self.pub.publish({...}));
#   - сохраняйте ссылку на задачу (например, self._tasks.append(task)),
#     чтобы сборщик мусора не убил её до завершения.


### ╰( ͡° ͜ʖ ͡° )つ──☆*:・ﾟ


Запуск:

In [ ]:
# Обновлённый main

async def main():
    ros = MiniROS()
    sensor = SensorNode(ros, frequency=1.5)
    logger = LoggerNode(ros)
    emergency = EmergencyStopNode(ros)  # <- новая нода

    task = asyncio.create_task(sensor.run())
    await asyncio.sleep(6)
    task.cancel()
    print("\n[Main] Завершение работы (shutdown).")

await main()

## Задача 3. Творческая.

Автор семинара устал придумывать задачи и их тестировать :)

**Придумайте самостоятельно задачу на основе класса MiniROS (или что-то прочее - по мере сил и возможностей). Обсудите её с преподавателем. Реализайте в каком-то виде. И продемонстрируйте перед аудиторией :) Если будут вопросы по коду / логике - будьте готовы ответить.**

Для начала в виде блок-схемы. С описанием логики. Реализовать можете в свободное от занятий время.

Удачи :)

Решение:

In [ ]:
### ╰( ͡° ͜ʖ ͡° )つ──☆*:・ﾟ


## 4. Задачки на асинхронность.

### Задача 4.1. Кофейня.

Вы открыли кофейню. У вас есть **два напитка**: эспрессо (готовится 2 секунды) и капучино (готовится 3 секунды). Каждый напиток - это отдельная асинхронная задача.

1. Напишите две корутины: `make_espresso()` и `make_cappuccino()`. Каждая:
   - печатает `"Начинаю готовить <название>..."`;
   - ждёт соответствующее время через `await asyncio.sleep(...)`;
   - печатает `"<Название> готов!"`.
2. Напишите `async def main()`, которая:
   - замеряет время через `time.time()`;
   - запускает **обе** корутины **параллельно** через `asyncio.gather(...)`;
   - печатает общее время.
3. Запустите `await main()`.

Ожидаемый результат: оба напитка готовятся **одновременно**. Общее время - **3 секунды** (по самому долгому), а не 5 (сумма).

Решение:

In [ ]:
import asyncio
import time

### ╰( ͡° ͜ʖ ͡° )つ──☆*:・ﾟ


### Задача 4.2. Скачивание файлов.

Вы пишете программу, которая скачивает **три файла** с сервера. Каждый файл скачивается своё время: 1, 2 и 3 секунды. Скачивание - это имитация через `await asyncio.sleep(...)`.

1. Напишите корутину `download_file(name, duration)`, которая:
   - печатает `"Начинаю скачивать <name>..."`;
   - ждёт `duration` секунд;
   - печатает `"<name> скачан за <duration> с"`.
2. Напишите `async def main()`, которая:
   - замеряет время;
   - запускает **все три** скачивания **параллельно** через `asyncio.gather`;
   - печатает общее время.
3. Запустите `await main()`.

Ожидаемый результат: Все три файла скачиваются одновременно. Общее время - **3 секунды** (самый долгий файл), а не 6 (сумма).

Решение:

In [ ]:
import asyncio
import time

### ╰( ͡° ͜ʖ ͡° )つ──☆*:・ﾟ


### Задача 4.3. Повар и официант.

В ресторане работают **повар** и **официант**. Повар готовит блюда и складывает их на стойку. Официант забирает блюда со стойки и относит гостям. Стойка вмещает **не больше 2 блюд** - если повар пытается положить третье, он ждёт, пока официант не заберёт хотя бы одно.

1. Используйте `asyncio.Queue(maxsize=2)` как "стойку".
2. Напишите корутину `chef(queue)`:
   - в цикле от 1 до 5 готовит блюдо (ждёт 0.5 с);
   - кладёт его в очередь через `await queue.put(f"Блюдо {i}")`;
   - печатает `"[Повар] положил Блюдо i"`.
3. Напишите корутину `waiter(queue)`:
   - в цикле 5 раз забирает блюдо через `await queue.get()`;
   - печатает `"[Официант] забрал <блюдо>"`;
   - «относит гостю» (ждёт 1 с);
   - печатает `"[Официант] отнёс <блюдо>"`.
4. Напишите `async def main()`, которая запускает **обе** корутины параллельно через `asyncio.gather`.
5. Запустите `await main()`.

Ожидаемый результат: повар и официант работают одновременно. Когда очередь заполнена (2 блюда), повар **ждёт**, пока официант не заберёт блюдо. В логах видно чередование: повар кладёт -> официант забирает -> повар кладёт снова.

Решение:

In [ ]:
import asyncio

### ╰( ͡° ͜ʖ ͡° )つ──☆*:・ﾟ


<h1>
<b>
Успехов :)
</h1>
</b>